# ScoreIA - Descricao Tecnica do Agente

Documento vivo com o estado implementado do agente ScoreIA.
Snapshot tecnico atualizado em: `2026-03-09`.


## Visao Geral

- Entrada principal de interacao: `main.py` e `tools/webapp/app.py`.
- Orquestrador central: `core/agent.py`.
- Memoria e governanca de conversas: `core/memory.py`.
- Camada de ML, interpretabilidade, relatorio e artefatos: `tools/`.
- Documentacao e log da fase: `docs/`.


## Funcionalidades Ativas

- Pipeline de ML com `train`, `test` e `oot`.
- Optuna/CV para `lightgbm`, `mlp` e `auto`.
- Threshold de classificacao otimizado por precision-recall.
- Interpretabilidade integrada:
  - LightGBM: feature importance + SHAP.
  - MLP: permutation importance + ALE.
- Feature selection temporal por estabilidade.
- Pipeline multiestagio de modelagem:
  - `model_full`,
  - `model_prunned`,
  - `model_reduced`.
- Exportacao completa de artefatos por estagio.
- Hash SHA-256 por artefato e assinatura consolidada por pacote.
- Interface web com mini-pipeline de validacao, persistencia do dataset e parametros avancados de modelagem.


## Inventario Tecnico Relevante

- `main.py`
- `core/agent.py`
- `core/memory.py`
- `tools/artifacts.py`
- `tools/artifact_export.py`
- `tools/data_pipeline.py`
- `tools/feature_postprocessing.py`
- `tools/feature_selection.py`
- `tools/interpretability.py`
- `tools/lightgbm_compat.py`
- `tools/metrics.py`
- `tools/ml_pipeline_flow.py`
- `tools/model_factory.py`
- `tools/model_training.py`
- `tools/optimization.py`
- `tools/reporting.py`
- `tools/runtime_config.py`
- `tools/webapp/app.py`
- `tools/webapp/storage.py`
- `tools/webapp/validation.py`


## Detalhamento por Arquivo

### `main.py`
- Entrada CLI do agente local.
- `run_cli()`: loop interativo com os comandos de estudo, planejamento, revisao de tarefas e abertura da modelagem.
- `handle_interacao(...)`: roteia o comando textual para o subsistema correto.
- `modelagem`: sobe a interface web do pipeline de ML e abre o navegador no endpoint local.

### `core/agent.py`
- `Agente.analisar_dados(config)`: orquestra a execucao ponta a ponta.
- Papel atual: manter a coordenacao de alto nivel e consolidar o retorno final do pipeline.
- Responsabilidades principais:
  - inicializar `run_context`,
  - carregar dados e preprocessing,
  - acionar feature selection temporal,
  - disparar `model_full`, `model_prunned` e `model_reduced` quando aplicavel,
  - consolidar `model_stage_outputs`,
  - registrar a execucao na memoria.

### `tools/ml_pipeline_flow.py`
- Modulo operacional do pipeline de ML.
- `inicializar_execucao_ml(...)`: cria o contexto temporal e a raiz da run.
- `carregar_pipeline_e_particionar(...)`: carrega dados, valida regras e monta `train/test/oot`.
- `aplicar_feature_selection_temporal_ml(...)`: executa selecao temporal por estabilidade.
- `aplicar_pos_processamento_features_ml(...)`: cria o postprocessor base do `model_full`.
- `derivar_postprocessor_pruning_ml(...)`: deriva o subconjunto do `model_prunned` por corte acumulado de importancia.
- `derivar_postprocessor_reducao_ml(...)`: deriva o `model_reduced` com PCA na cauda menos importante.
- `executar_estagio_modelagem_ml(...)`: executor unico de estagio, incluindo CV/Optuna, treino final, metricas, interpretabilidade, relatorio e exportacao.
- `gerar_dataset_scored_ml(...)`: gera scored dataset do estagio.
- `gerar_relatorio_ml(...)`: renderiza e salva HTML do estagio.

### `tools/data_pipeline.py`
- `validar_temporal_feature(...)`: valida formato `YYYYMMDD` ou `YYYYMM`.
- `split_holdout(...)`: cria `train`, `test` e `oot`, exclui a variavel temporal do ajuste e aplica estrategia de segmento.
- `fit_transform(...)` / `transform(...)`: preprocessing reutilizavel com imputacao opcional.

### `tools/feature_postprocessing.py`
- `FeaturePostProcessor`: estrutura encadeavel de pos-processamento de features.
- Suporta:
  - selecao de subconjuntos,
  - encadeamento via `upstream`,
  - PCA nas features menos importantes,
  - serializacao para reproducibilidade.

### `tools/model_training.py`
- `train_single_model(...)`: treino final de LightGBM ou MLP.
- `train_bucket_ensemble(...)`: treino segmentado por bucket para LightGBM.
- `predict_model_outputs(...)`: padroniza predicoes de `train/test/oot`.
- `summarize_feature_importance_profile(...)`: gera ranking, conjunto ativo e corte acumulado de importancia.

### `tools/lightgbm_compat.py`
- `prepare_lightgbm_predict_input(...)`: converte `ndarray` para `DataFrame` nomeado quando necessario, evitando warnings de `feature names` no LightGBM/sklearn.

### `tools/runtime_config.py`
- `configure_runtime_environment()`: aplica configuracao defensiva de runtime para o ecossistema `joblib/loky` e filtros de warnings conhecidos sem impacto funcional.

### `tools/optimization.py`
- `_objective_factory(...)`: objetivo Optuna por modelo/estagio.
- `otimizar_modelo_cv(...)`: executa estudos `single-model` ou `auto`.

### `tools/interpretability.py`
- `gerar_interpretabilidade(...)`: monta importance, SHAP ou ALE do estagio.
- `_tree_shap_summary(...)`: SHAP para modelos de arvore com fallback `pred_contrib`.

### `tools/reporting.py`
- `gerar_relatorio_tecnico_html(...)`: compoe o HTML tecnico do estagio.
- Inclui graficos Optuna, curva de aprendizado, Top 20, SHAP/ALE e analise temporal/segmentativa.
- O grafico temporal principal usa eixo duplo: barras para volume e linhas para media do target.

### `tools/artifact_export.py`
- `exportar_artefatos_execucao(...)`: exporta o pacote completo do estagio.
- Gera `model`, `preprocess`, `inference`, `metadata`, `reports`, `scored_data`, arquivos de ambiente, hashes e manifest.

### `tools/webapp/app.py`
- Endpoint principal da interface web.
- Diferencia `validate_only` e `train_model`.
- Persiste o dataset validado para reuso no treino.
- Exponde parametros de temporalidade, segmentacao, feature selection temporal, pruning, PCA da cauda e balanceamento.

### `tools/webapp/validation.py`
- Mini-pipeline de validacao do dataset para a interface web.
- Resolve validacao temporal, limite de folds por meses in-time e mensagens ao usuario.

### `tools/webapp/storage.py`
- Persistencia temporaria dos uploads web.
- Limpeza da pasta `web_uploads` ao final da execucao completa.


## Contrato de Saida

- `artifacts/run_<timestamp>/model_full/...`
- `artifacts/run_<timestamp>/model_prunned/...` quando pruning gera reducao real.
- `artifacts/run_<timestamp>/model_reduced/...` quando a reducao PCA e aplicavel.
- Cada estagio possui:
  - modelo exportado,
  - preprocessamento serializado,
  - script de inferencia,
  - relatorio HTML,
  - interpretabilidade JSON,
  - scored dataset,
  - metadata,
  - hashes e manifest.
- O retorno consolidado do agente aponta o `final_stage_name` e expande `model_stage_outputs` com caminhos e resumo de cada estagio executado.
